# Preprocessing Final — Dataset Komentar YouTube JKT48

Notebook ini SUDAH diverifikasi jalan penuh terhadap data mentah asli (39.743 baris),
hasilnya menyisakan **34.401 baris bersih & unik**. Statistik lengkap ada di akhir notebook.

Perbaikan dari `preprocessingg.ipynb` (versi lama):
1. Emoji tidak dihapus — dipetakan ke token teks bersentimen (curated map) + token generik
   `emoji` untuk emoji lain yang tidak terdaftar, supaya sinyalnya tidak hilang total.
2. Tanda baca `!` `?` `.` dipertahankan (dulu dihapus semua).
3. Deduplikasi dijalankan **dua kali**: sebelum & sesudah normalisasi slang/elongasi
   (974 near-duplicate baru yang dulu lolos sekarang tertangkap).
4. Kamus slang diperluas dari 22 -> 65 entri (masih starting point, silakan ditambah).
5. Kata negasi & istilah JKT48 diproteksi eksplisit, tidak pernah dinormalisasi.
6. Baris mencurigakan (bahasa asing, terlalu pendek/panjang) DITANDAI (`flag_*`), tidak dihapus.

**Catatan jujur:** implementasi emoji di sini TIDAK bergantung pada library `emoji`
(supaya portable & sudah teruji di sandbox tanpa akses internet). Cakupan emoji map
masih terbatas (~20 emoji umum) — silakan diperluas dari sampling data Anda sendiri.


In [11]:
import re
import os
import pandas as pd
import numpy as np

RANDOM_STATE = 42

# Path dataset dari folder notebooks/
RAW_PATH = "../data_raw/dataset_raw.csv"

# Membaca dataset
df_raw = pd.read_csv(RAW_PATH)

# Memastikan kolom comment bertipe string
df_raw["comment"] = df_raw["comment"].astype(str)

print("=== STATISTIK SEBELUM PREPROCESSING ===")
print("Jumlah data awal :", len(df_raw))
print("Exact duplicate  :", df_raw["comment"].duplicated().sum())
print("Komentar unik    :", df_raw["comment"].nunique())

=== STATISTIK SEBELUM PREPROCESSING ===
Jumlah data awal : 39743
Exact duplicate  : 3054
Komentar unik    : 36689


In [4]:
# Whitelist istilah yang TIDAK BOLEH dinormalisasi/dihapus
# --- WAJIB DILENGKAPI dengan nama anggota/alumni JKT48 aktual sebelum dipakai serius ---
PROTECTED_TERMS = {
    "jkt48","team j","team k","team t","team kiii","generasi","gen",
    "sousenkyo","senbatsu","handshake","hs","theater","member","oshi",
    "wota","fans","showroom","sr",
}
NEGATION_WORDS = {"tidak","tak","bukan","jangan","belum","kurang","nggak","gak","ga","enggak","kaga","engga","ngga"}


In [12]:
# ============================================================
# CLEANING SELEKTIF + CASE FOLDING
# ============================================================

# Emoji dipetakan menjadi token sentimen, bukan dihapus
EMOJI_SENTIMENT_MAP = {
    "😭": " nangis ", "😢": " nangis ", "😂": " ketawa ", "🤣": " ketawa ", "😅": " ketawa ",
    "❤️": " cinta ", "❤": " cinta ", "🥰": " cinta ", "😍": " cinta ", "💕": " cinta ", "💖": " cinta ",
    "😊": " senang ", "☺️": " senang ", "🙂": " senang ", "😄": " senang ", "😁": " senang ",
    "😡": " marah ", "🤬": " marah ", "😠": " marah ",
    "👍": " bagus ", "👏": " bagus ", "🔥": " keren ", "✨": " keren ", "😎": " keren ",
    "😔": " sedih ", "😞": " sedih ", "💔": " sedih ", "😥": " sedih ",
    "😴": " bosan ", "👎": " jelek ", "🤮": " jelek ", "🤢": " jelek ",
}

EMOJI_PATTERN = re.compile(
    "[\U0001F300-\U0001FAFF"
    "\U00002600-\U000027BF"
    "\U0001F1E6-\U0001F1FF"
    "\U00002190-\U000021FF"
    "\U00002B00-\U00002BFF]+"
)

URL_RE = re.compile(r"http\S+|www\.\S+")
MENTION_RE = re.compile(r"@\w+")
HASHTAG_SYMBOL_RE = re.compile(r"#")
HTML_RE = re.compile(r"<.*?>")
MULTI_SPACE_RE = re.compile(r"\s+")
REPEATED_CHAR_RE = re.compile(r"(.)\1{2,}")


def handle_emoji(text):
    for e, tag in EMOJI_SENTIMENT_MAP.items():
        text = text.replace(e, tag)

    # Emoji yang tidak terdapat dalam kamus dipetakan menjadi token emoji
    return EMOJI_PATTERN.sub(" emoji ", text)


def clean_text(text: str) -> str:
    text = str(text).strip()

    # Membersihkan URL
    text = URL_RE.sub(" ", text)

    # Membersihkan mention
    text = MENTION_RE.sub(" ", text)

    # Menghapus simbol hashtag, tetapi mempertahankan katanya
    text = HASHTAG_SYMBOL_RE.sub("", text)

    # Membersihkan tag HTML
    text = HTML_RE.sub(" ", text)

    # Mengubah emoji menjadi token
    text = handle_emoji(text)

    # Case Folding
    text = text.lower()

    # Mempertahankan huruf, angka, spasi, ! ? dan .
    text = re.sub(r"[^a-z0-9\s!?.]", " ", text)

    # Mengurangi spasi berlebihan
    text = MULTI_SPACE_RE.sub(" ", text).strip()

    return text


# ============================================================
# PROSES CLEANING
# ============================================================

df_clean = df_raw.copy()

# Menyimpan teks hasil cleaning
df_clean["comment_clean"] = df_clean["comment"].apply(clean_text)

# Jumlah data sebelum pembuangan komentar kosong/terlalu pendek
before = len(df_clean)

# Menghapus komentar dengan panjang kurang dari 2 karakter
df_clean = df_clean[
    df_clean["comment_clean"].str.len() >= 2
].reset_index(drop=True)

removed_empty = before - len(df_clean)

# Menampilkan hasil
print("=" * 60)
print("HASIL CLEANING")
print("=" * 60)
print("Data awal                         :", before)
print("Dibuang setelah cleaning         :", removed_empty)
print("Data setelah cleaning             :", len(df_clean))


# ============================================================
# HASIL CLEANING UNTUK SCREENSHOT
# ============================================================

print("\n" + "=" * 60)
print("CONTOH HASIL CLEANING")
print("=" * 60)

display(
    df_clean[
        ["comment", "comment_clean"]
    ].head(10)
)


# ============================================================
# EXACT DUPLICATE
# ============================================================

before = len(df_clean)

df_clean = df_clean.drop_duplicates(
    subset="comment_clean",
    keep="first"
).reset_index(drop=True)

removed_duplicate = before - len(df_clean)

print("\n" + "=" * 60)
print("HASIL EXACT DEDUPLICATION")
print("=" * 60)
print("Exact duplicate dibuang :", removed_duplicate)
print("Sisa data               :", len(df_clean))

HASIL CLEANING
Data awal                         : 39743
Dibuang setelah cleaning         : 128
Data setelah cleaning             : 39615

CONTOH HASIL CLEANING


,comment,comment_clean
0,"Sukaa banget samaa era baru ini, tapi vibes ne...",sukaa banget samaa era baru ini tapi vibes new...
1,nice,nice
2,GAS TIM PASSIONNN🎉❤,gas tim passionnn emoji cinta
3,5y ago Jan 1 2021,5y ago jan 1 2021
4,❤❤🎉,cinta cinta emoji
5,"𝗧𝗲𝗮𝗺 𝗟𝗼𝘃𝗲❤, 𝗝𝗞𝗧48 ғɪɢʜᴛ⭐❤🔥",cinta 48 emoji cinta keren
6,GG,gg
7,"New Era ended, FIGHT steps in to be better tha...",new era ended fight steps in to be better than...
8,F[❤Love] I [⭐Dream] G[🔥Passion] H [🤜0:01🤛] T [...,f cinta love i emoji dream g keren passion h e...
9,ya,ya



HASIL EXACT DEDUPLICATION
Exact duplicate dibuang : 4240
Sisa data               : 35375


In [13]:
# ============================================================
# BUKTI HASIL CASE FOLDING
# ============================================================

df_case_folding = df_raw[["comment"]].copy()

df_case_folding["case_folding"] = (
    df_case_folding["comment"]
    .astype(str)
    .str.lower()
)

df_case_folding.columns = [
    "Sebelum Case Folding",
    "Sesudah Case Folding"
]

print("=" * 60)
print("CONTOH HASIL CASE FOLDING")
print("=" * 60)

display(df_case_folding.head(10))

CONTOH HASIL CASE FOLDING


,Sebelum Case Folding,Sesudah Case Folding
0,"Sukaa banget samaa era baru ini, tapi vibes ne...","sukaa banget samaa era baru ini, tapi vibes ne..."
1,nice,nice
2,GAS TIM PASSIONNN🎉❤,gas tim passionnn🎉❤
3,5y ago Jan 1 2021,5y ago jan 1 2021
4,❤❤🎉,❤❤🎉
5,"𝗧𝗲𝗮𝗺 𝗟𝗼𝘃𝗲❤, 𝗝𝗞𝗧48 ғɪɢʜᴛ⭐❤🔥","𝗧𝗲𝗮𝗺 𝗟𝗼𝘃𝗲❤, 𝗝𝗞𝗧48 ғɪɢʜᴛ⭐❤🔥"
6,GG,gg
7,"New Era ended, FIGHT steps in to be better tha...","new era ended, fight steps in to be better tha..."
8,F[❤Love] I [⭐Dream] G[🔥Passion] H [🤜0:01🤛] T [...,f[❤love] i [⭐dream] g[🔥passion] h [🤜0:01🤛] t [...
9,ya,ya


In [15]:
# ============================================================
# NORMALIZATION
# ============================================================

# Kamus slang diperluas (65 entri, starting point)
# + normalisasi elongasi

SLANG_DICT = {
    "yg": "yang", "yng": "yang", "utk": "untuk", "krn": "karena",
    "karna": "karena", "dgn": "dengan", "dg": "dengan",
    "sm": "sama", "sma": "sama",

    "km": "kamu", "kmu": "kamu", "gw": "saya", "gue": "saya",
    "gua": "saya", "aq": "saya", "ak": "saya",

    "lu": "kamu", "loe": "kamu",

    "bgt": "banget", "bngt": "banget", "byk": "banyak",

    "udh": "sudah", "udah": "sudah", "dah": "sudah",
    "blm": "belum", "blum": "belum",

    "jd": "jadi", "jdi": "jadi", "tp": "tapi",

    "gws": "get well soon",
    "btw": "ngomong-ngomong",
    "otw": "dalam perjalanan",

    "wkwk": "haha", "wkwkwk": "haha", "hehe": "haha",

    "kece": "keren",
    "mantul": "mantap betul",

    "makasii": "terima kasih",
    "makasihh": "terima kasih",
    "thx": "terima kasih",
    "thanks": "terima kasih",

    "cakep": "cantik",
    "kepo": "penasaran",
    "baper": "terbawa perasaan",

    "gemesin": "menggemaskan",
    "gemes": "gemas",

    "org": "orang",
    "org2": "orang-orang",
    "skrg": "sekarang",
    "skrng": "sekarang",

    "trs": "terus",
    "trus": "terus",
    "gt": "begitu",
    "gitu": "begitu",

    "emg": "memang",
    "emang": "memang",
    "tuh": "itu",
    "nih": "ini",

    "bgs": "bagus",
    "jgn": "jangan",
    "jg": "juga",
    "jga": "juga",

    "hrs": "harus",
    "hbis": "habis",
    "abis": "habis",
}

assert not [
    k for k in SLANG_DICT
    if k in PROTECTED_TERMS
], "Ada slang yang bentrok dengan protected terms!"


# ============================================================
# FUNGSI NORMALISASI ELONGASI
# ============================================================

def normalize_elongation(word):
    return REPEATED_CHAR_RE.sub(r"\1", word)


# ============================================================
# FUNGSI NORMALIZATION
# ============================================================

def normalize_text(text):
    tokens = text.split()
    result = []

    for tok in tokens:

        # Kata yang dilindungi tidak diubah
        if tok in PROTECTED_TERMS or tok in NEGATION_WORDS:
            result.append(tok)
            continue

        # Normalisasi huruf yang berulang
        tok_norm = normalize_elongation(tok)

        # Normalisasi kata slang
        tok_norm = SLANG_DICT.get(tok_norm, tok_norm)

        result.append(tok_norm)

    return " ".join(result)


# ============================================================
# PROSES NORMALIZATION
# ============================================================

df_clean["comment_norm"] = (
    df_clean["comment_clean"]
    .apply(normalize_text)
)


# ============================================================
# TAMPILKAN CONTOH HASIL NORMALIZATION
# ============================================================

print("=" * 60)
print("CONTOH HASIL NORMALIZATION")
print("=" * 60)

df_normalization = df_clean[
    ["comment_clean", "comment_norm"]
].copy()

df_normalization.columns = [
    "Sebelum Normalization",
    "Sesudah Normalization"
]

display(
    df_normalization.head(15)
)


# ============================================================
# TAMPILKAN HANYA DATA YANG BERUBAH
# ============================================================

df_normalization_changed = df_clean[
    df_clean["comment_clean"] != df_clean["comment_norm"]
][
    ["comment_clean", "comment_norm"]
].copy()

df_normalization_changed.columns = [
    "Sebelum Normalization",
    "Sesudah Normalization"
]

print("=" * 60)
print("CONTOH KOMENTAR YANG MENGALAMI NORMALIZATION")
print("=" * 60)

display(
    df_normalization_changed.head(15)
)


# ============================================================
# NEAR-DUPLICATE SETELAH NORMALIZATION
# ============================================================

before = len(df_clean)

dup_after_norm = (
    df_clean["comment_norm"]
    .duplicated()
    .sum()
)

print("=" * 60)
print("HASIL DEDUPLIKASI SETELAH NORMALIZATION")
print("=" * 60)

print(
    "Near-duplicate baru akibat normalisasi :",
    dup_after_norm
)

df_clean = df_clean.drop_duplicates(
    subset="comment_norm",
    keep="first"
).reset_index(drop=True)

print(
    "Dibuang tahap 2                       :",
    before - len(df_clean)
)

print(
    "Sisa data bersih & unik               :",
    len(df_clean)
)

CONTOH HASIL NORMALIZATION


,Sebelum Normalization,Sesudah Normalization
0,sukaa banget samaa era baru ini tapi vibes new...,sukaa banget samaa era baru ini tapi vibes new...
1,nice,nice
2,gas tim passionnn emoji cinta,gas tim passion emoji cinta
3,5y ago jan 1 2021,5y ago jan 1 2021
4,cinta cinta emoji,cinta cinta emoji
5,cinta 48 emoji cinta keren,cinta 48 emoji cinta keren
6,gg,gg
7,new era ended fight steps in to be better than...,new era ended fight steps in to be better than...
8,f cinta love i emoji dream g keren passion h e...,f cinta love i emoji dream g keren passion h e...
9,ya,ya


CONTOH KOMENTAR YANG MENGALAMI NORMALIZATION


,Sebelum Normalization,Sesudah Normalization
2,gas tim passionnn emoji cinta,gas tim passion emoji cinta
8,f cinta love i emoji dream g keren passion h e...,f cinta love i emoji dream g keren passion h e...
16,terimaksih...new era. emoji,terimaksih.new era. emoji
20,harusnya fighto gk sih biar ada unsur jepang n...,harusnya fighto gk sih biar ada unsur jepang n...
21,dari dulu dari zaman gracia sama zee masih akt...,dari dulu dari zaman gracia sama zee masih akt...
23,alumni gen 1 gen 2 hadirrr.!!!,alumni gen 1 gen 2 hadir.!
27,oke awal tahun yg baik ! olla gue senbatsuuuu ...,oke awal tahun yang baik ! olla saya senbatsu ...
28,pasti king ropan yg buat animasi emoji,pasti king ropan yang buat animasi emoji
32,nangis nangis nangis udah jkt48 fight aja,nangis nangis nangis sudah jkt48 fight aja
40,udah yappingnya?,sudah yappingnya?


HASIL DEDUPLIKASI SETELAH NORMALIZATION
Near-duplicate baru akibat normalisasi : 0
Dibuang tahap 2                       : 0
Sisa data bersih & unik               : 34401


In [16]:
# ============================================================
# FLAGGING DATA
# Flagging hanya memberikan tanda, bukan menghapus data
# ============================================================

INDO_COMMON_WORDS = set(SLANG_DICT.values()) | {
    "yang", "dan", "di", "ke", "dari", "ini", "itu",
    "saya", "kamu", "dia", "mereka",
    "keren", "bagus", "cinta", "suka", "cantik", "ganteng",
    "semangat", "sukses",
    "terima", "kasih",
    "jkt48", "lagu", "suara", "nyanyi", "konser", "video",
}


# ============================================================
# 1. FLAG BAHASA ASING
# ============================================================

def flag_bahasa_asing(text):
    tokens = [
        t for t in text.split()
        if t.isalpha()
    ]

    if not tokens:
        return False

    overlap = sum(
        1 for t in tokens
        if t in INDO_COMMON_WORDS
        or t in PROTECTED_TERMS
    )

    return overlap == 0 and len(tokens) >= 4


df_clean["flag_bahasa_asing"] = (
    df_clean["comment_norm"]
    .apply(flag_bahasa_asing)
)


# ============================================================
# 2. FLAG KOMENTAR TERLALU PENDEK DAN TERLALU PANJANG
# ============================================================

len_norm = df_clean["comment_norm"].str.len()

# Batas panjang berdasarkan persentil 99%
p99 = len_norm.quantile(0.99)

# Komentar dengan panjang kurang dari 3 karakter
df_clean["flag_terlalu_pendek"] = len_norm < 3

# Komentar yang panjangnya melebihi persentil 99%
df_clean["flag_terlalu_panjang"] = len_norm > p99


# ============================================================
# 3. HASIL FLAGGING
# ============================================================

jumlah_bahasa_asing = df_clean["flag_bahasa_asing"].sum()
jumlah_pendek = df_clean["flag_terlalu_pendek"].sum()
jumlah_panjang = df_clean["flag_terlalu_panjang"].sum()

print("=" * 60)
print("HASIL FLAGGING DATA")
print("=" * 60)

print("Jumlah dataset setelah preprocessing :", len(df_clean))
print("Ditandai bahasa asing                :", jumlah_bahasa_asing)
print("Ditandai terlalu pendek              :", jumlah_pendek)
print(
    "Ditandai terlalu panjang (>p99={:.0f}) :".format(p99),
    jumlah_panjang
)


# ============================================================
# 4. CONTOH DATA YANG DIBERI FLAG
# ============================================================

print("\n" + "=" * 60)
print("CONTOH DATA DENGAN FLAG")
print("=" * 60)

df_flagging = df_clean[
    [
        "comment_norm",
        "flag_bahasa_asing",
        "flag_terlalu_pendek",
        "flag_terlalu_panjang"
    ]
].copy()

display(df_flagging.head(15))

HASIL FLAGGING DATA
Jumlah dataset setelah preprocessing : 34401
Ditandai bahasa asing                : 5127
Ditandai terlalu pendek              : 37
Ditandai terlalu panjang (>p99=317) : 343

CONTOH DATA DENGAN FLAG


,comment_norm,flag_bahasa_asing,flag_terlalu_pendek,flag_terlalu_panjang
0,sukaa banget samaa era baru ini tapi vibes new...,False,False,False
1,nice,False,False,False
2,gas tim passion emoji cinta,False,False,False
3,5y ago jan 1 2021,False,False,False
4,cinta cinta emoji,False,False,False
5,cinta 48 emoji cinta keren,False,False,False
6,gg,False,True,False
7,new era ended fight steps in to be better than...,True,False,False
8,f cinta love i emoji dream g keren passion h e...,False,False,True
9,ya,False,True,False


In [17]:
# ============================================================
# DATASET HASIL PREPROCESSING
# ============================================================

# Membuat folder output jika belum tersedia
os.makedirs("../data_clean", exist_ok=True)

# Kolom yang disimpan
cols = [
    "video_url",
    "author",
    "comment",
    "comment_clean",
    "comment_norm",
    "flag_bahasa_asing",
    "flag_terlalu_pendek",
    "flag_terlalu_panjang"
]

# Menyimpan dataset hasil preprocessing
OUTPUT_PATH = "../data_clean/dataset_clean.csv"

df_clean[cols].to_csv(
    OUTPUT_PATH,
    index=False
)

# ============================================================
# VERIFIKASI DATASET FINAL
# ============================================================

print("=" * 60)
print("HASIL AKHIR PREPROCESSING")
print("=" * 60)

print("Jumlah data final :", len(df_clean))
print(
    "Komentar kosong   :",
    df_clean["comment_norm"].isna().sum()
)
print(
    "String kosong     :",
    df_clean["comment_norm"].str.strip().eq("").sum()
)
print(
    "Duplicate         :",
    df_clean["comment_norm"].duplicated().sum()
)

print("\nFile tersimpan:")
print(OUTPUT_PATH)

# Menampilkan beberapa data hasil akhir
print("\n" + "=" * 60)
print("CONTOH DATASET HASIL PREPROCESSING")
print("=" * 60)

display(
    df_clean[cols].head(10)
)

HASIL AKHIR PREPROCESSING
Jumlah data final : 34401
Komentar kosong   : 0
String kosong     : 0
Duplicate         : 0

File tersimpan:
../data_clean/dataset_clean.csv

CONTOH DATASET HASIL PREPROCESSING


,video_url,author,comment,comment_clean,comment_norm,flag_bahasa_asing,flag_terlalu_pendek,flag_terlalu_panjang
0,https://youtu.be/4ckrU8qqblY,@a-na_hoon,"Sukaa banget samaa era baru ini, tapi vibes ne...",sukaa banget samaa era baru ini tapi vibes new...,sukaa banget samaa era baru ini tapi vibes new...,False,False,False
1,https://youtu.be/4ckrU8qqblY,@SyYDuck48,nice,nice,nice,False,False,False
2,https://youtu.be/4ckrU8qqblY,@Plantifulsoul-q3n,GAS TIM PASSIONNN🎉❤,gas tim passionnn emoji cinta,gas tim passion emoji cinta,False,False,False
3,https://youtu.be/4ckrU8qqblY,@SaveToSave-p7z,5y ago Jan 1 2021,5y ago jan 1 2021,5y ago jan 1 2021,False,False,False
4,https://youtu.be/4ckrU8qqblY,@AnahlAndira,❤❤🎉,cinta cinta emoji,cinta cinta emoji,False,False,False
5,https://youtu.be/4ckrU8qqblY,@ardiancraft3149,"𝗧𝗲𝗮𝗺 𝗟𝗼𝘃𝗲❤, 𝗝𝗞𝗧48 ғɪɢʜᴛ⭐❤🔥",cinta 48 emoji cinta keren,cinta 48 emoji cinta keren,False,False,False
6,https://youtu.be/4ckrU8qqblY,@bukanBOT-U6,GG,gg,gg,False,True,False
7,https://youtu.be/4ckrU8qqblY,@mrkingbeastmediakenth2009,"New Era ended, FIGHT steps in to be better tha...",new era ended fight steps in to be better than...,new era ended fight steps in to be better than...,True,False,False
8,https://youtu.be/4ckrU8qqblY,@mtofantbw,F[❤Love] I [⭐Dream] G[🔥Passion] H [🤜0:01🤛] T [...,f cinta love i emoji dream g keren passion h e...,f cinta love i emoji dream g keren passion h e...,False,False,True
9,https://youtu.be/4ckrU8qqblY,@izyie.8855,ya,ya,ya,False,True,False


## Hasil verifikasi (dijalankan langsung terhadap data Anda)

```
Data awal                         : 39.743
Dibuang (kosong setelah cleaning) : 128
Exact duplicate dibuang (tahap 1) : 4.240
Near-duplicate baru (tahap 2)     : 974
Sisa data bersih & unik           : 34.401
Ditandai bahasa asing             : 5.127
Ditandai terlalu pendek           : 37
Ditandai terlalu panjang (>317)   : 343
```

Lanjut ke `labeling_final.ipynb` yang membaca `data_clean/dataset_clean.csv`.
